# HoVer-Net checkpoint — which PanNuke fold is held out? (memorization-gap test)

The released `hovernet_fast_pannuke_type` checkpoint has **no training metadata**
(only a weight dict), so we don't know its train/test fold split. Using it on a fold
it trained on = **leakage**. Here we find the held-out fold empirically: run on
Fold 1/2/3 and compare per-fold **counting MAE + foreground Dice**. The fold the model
is clearly **worst** on is the held-out (safe) one.

**Decision:** if **Fold 3** is clearly worst → safe to use (it's our conformal set).
If all folds look equally good → trained on all → leaky → must retrain on Fold 1+2.

**Attach:** `hipinhththu/pannuke` + the dataset holding the checkpoint file
(`hovernet_fast_pannuke_type_tf2pytorch`). GPU T4.

## 00 — Setup: clone vqdang/hover_net, deps, find checkpoint

In [ ]:
import subprocess, sys, os, glob, time, json
import numpy as np
import torch
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

WORK = "/kaggle/working"
HOVER_DIR = f"{WORK}/hover_net"
if not os.path.isdir(HOVER_DIR):
    subprocess.run(["git", "clone", "https://github.com/vqdang/hover_net.git", HOVER_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scipy", "scikit-image", "opencv-python-headless"], check=True)

# locate the checkpoint file (no extension, ~150MB) anywhere under /kaggle/input
def find_ckpt():
    pats = ["/kaggle/input/**/hovernet_fast_pannuke_type*",
            "/kaggle/input/**/*hovernet*pannuke*"]
    for p in pats:
        for h in glob.glob(p, recursive=True):
            if os.path.isfile(h) and os.path.getsize(h) > 50_000_000:
                return h
    return None
CKPT = find_ckpt()
assert CKPT, "HoVer-Net checkpoint not found - attach the dataset holding it"
print("Checkpoint:", CKPT, f"({os.path.getsize(CKPT)/1e6:.0f} MB)")

if HOVER_DIR not in sys.path:
    sys.path.insert(0, HOVER_DIR)

## Helper modules (PanNuke loader + metrics)

In [ ]:
%%writefile pannuke_loader.py
from __future__ import annotations
from pathlib import Path
from typing import Iterator, List, Optional, Tuple
import numpy as np

CELL_TYPES: List[str] = [
    "Neoplastic", "Inflammatory", "Connective", "Dead", "Epithelial",
]

DEFAULT_ROOT = Path("/kaggle/input/datasets/hipinhththu/pannuke")

def _to_uint8(arr: np.ndarray) -> np.ndarray:
    if arr.dtype == np.uint8:
        return arr
    if arr.max() <= 1.0:
        return (arr * 255).round().clip(0, 255).astype(np.uint8)
    return arr.clip(0, 255).astype(np.uint8)

class PanNukeFold:

    def __init__(self, root, fold: int):
        self.root = Path(root)
        self.fold = fold
        sub = f"Fold {fold}"
        f = f"fold{fold}"
        candidates = [
            self.root / f / sub,   
            self.root / sub,        
        ]
        base = next((c for c in candidates if (c / "images" / f / "images.npy").exists()),
                    None)
        if base is None:
            raise FileNotFoundError(
                f"Không tìm thấy Fold {fold} ở: " + " hoặc ".join(str(c) for c in candidates)
            )
        self.images_path = base / "images" / f / "images.npy"
        self.masks_path  = base / "masks"  / f / "masks.npy"

        
        type_candidates = [
            base / "images" / f / "types.npy",   
            base / "types" / f / "types.npy",     
            base / "types.npy",
            self.root / f / "types.npy",
            self.root / "types" / f / "types.npy",
        ]
        self.types_path = next((p for p in type_candidates if p.exists()), None)

        for p in (self.images_path, self.masks_path):
            if not p.exists():
                raise FileNotFoundError(f"Missing: {p}")

        self.images = np.load(self.images_path, mmap_mode="r")
        self.masks  = np.load(self.masks_path,  mmap_mode="r")
        if self.types_path is not None:
            self.tissue_types = np.load(self.types_path, allow_pickle=True)
        else:
            n = self.images.shape[0]
            self.tissue_types = np.array(["unknown"] * n, dtype=object)
            print(f"[Fold {fold}] WARN: types.npy không tìm thấy ở:")
            for c in type_candidates:
                print(f"  - {c}")
            print(f"[Fold {fold}] Dùng placeholder 'unknown' cho {n} ảnh.")

        assert self.images.shape[0] == self.masks.shape[0] == self.tissue_types.shape[0]
        assert self.images.shape[1:] == (256, 256, 3), f"unexpected: {self.images.shape}"
        assert self.masks.shape[1:]  == (256, 256, 6), f"unexpected: {self.masks.shape}"

    def __len__(self) -> int:
        return self.images.shape[0]

    def __getitem__(self, idx: int) -> dict:
        img = _to_uint8(np.array(self.images[idx]))
        m_all = np.array(self.masks[idx], dtype=np.int32)
        masks_per_type = m_all[..., :5].transpose(2, 0, 1)
        counts = np.array(
            [int(np.unique(masks_per_type[k]).size - 1) for k in range(5)],
            dtype=np.int32,
        )
        return {
            "image": img,
            "masks": masks_per_type,
            "counts": counts,
            "tissue": str(self.tissue_types[idx]),
            "fold": self.fold,
            "idx": idx,
        }

    def iter_samples(self, start: int = 0, end: Optional[int] = None) -> Iterator[dict]:
        end = end or len(self)
        for i in range(start, end):
            yield self[i]

def load_all_folds(root=DEFAULT_ROOT) -> Tuple[PanNukeFold, PanNukeFold, PanNukeFold]:
    root = Path(root)
    return PanNukeFold(root, 1), PanNukeFold(root, 2), PanNukeFold(root, 3)

In [ ]:
%%writefile metrics.py
from __future__ import annotations
from typing import Sequence
import numpy as np

def binary_iou(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0

def binary_dice(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    sa, sb = a.sum(), b.sum()
    return float(2 * inter) / float(sa + sb) if (sa + sb) > 0 else 0.0

def match_pred_to_gt(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray],
                     iou_thresh: float = 0.5) -> dict:
    if not pred_masks and not gt_masks:
        return {"tp": 0, "fp": 0, "fn": 0, "mean_iou": 0.0}
    if not pred_masks:
        return {"tp": 0, "fp": 0, "fn": len(gt_masks), "mean_iou": 0.0}
    if not gt_masks:
        return {"tp": 0, "fp": len(pred_masks), "fn": 0, "mean_iou": 0.0}

    iou_matrix = np.zeros((len(pred_masks), len(gt_masks)), dtype=np.float32)
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            iou_matrix[i, j] = binary_iou(pm, gm)

    matched_pred, matched_gt = set(), set()
    ious = []
    pairs = np.dstack(np.unravel_index(np.argsort(-iou_matrix.ravel()), iou_matrix.shape))[0]
    for i, j in pairs:
        if iou_matrix[i, j] < iou_thresh:
            break
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(int(i)); matched_gt.add(int(j))
        ious.append(float(iou_matrix[i, j]))

    tp = len(matched_pred)
    fp = len(pred_masks) - tp
    fn = len(gt_masks)  - len(matched_gt)
    return {"tp": tp, "fp": fp, "fn": fn,
            "mean_iou": float(np.mean(ious)) if ious else 0.0}

def panoptic_quality(stats: dict) -> dict:
    tp, fp, fn = stats["tp"], stats["fp"], stats["fn"]
    sq = stats["mean_iou"]
    denom = tp + 0.5 * fp + 0.5 * fn
    rq = tp / denom if denom > 0 else 0.0
    pq = sq * rq
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"PQ": pq, "SQ": sq, "RQ": rq, "F1": f1, "P": precision, "R": recall}

def aggregate_iou_image(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray]) -> float:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu)

def aggregate_iou_dice_image(pred_masks: Sequence[np.ndarray],
                              gt_masks: Sequence[np.ndarray]) -> tuple:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu), binary_dice(pu, gu)

def union_masks(masks: Sequence[np.ndarray], shape=(256, 256)) -> np.ndarray:
    u = np.zeros(shape, dtype=bool)
    for m in masks:
        u |= m.astype(bool)
    return u.astype(np.uint8)

class ClassWiseAccumulator:

    def __init__(self, class_names):
        self.class_names = list(class_names)
        self.tp = {c: 0 for c in self.class_names}
        self.fp = {c: 0 for c in self.class_names}
        self.fn = {c: 0 for c in self.class_names}

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray, class_name: str):
        p = pred_mask.astype(bool)
        g = gt_mask.astype(bool)
        self.tp[class_name] += int(np.logical_and(p, g).sum())
        self.fp[class_name] += int(np.logical_and(p, np.logical_not(g)).sum())
        self.fn[class_name] += int(np.logical_and(np.logical_not(p), g).sum())

    def class_iou(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = tp + fp + fn
        return float(tp) / float(denom) if denom > 0 else 0.0

    def class_dice(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = 2 * tp + fp + fn
        return float(2 * tp) / float(denom) if denom > 0 else 0.0

    def mIoU(self) -> float:
        return float(np.mean([self.class_iou(c) for c in self.class_names]))

    def mDice(self) -> float:
        return float(np.mean([self.class_dice(c) for c in self.class_names]))

    def summary(self) -> dict:
        per_class = {c: {"IoU": self.class_iou(c), "Dice": self.class_dice(c),
                          "TP": self.tp[c], "FP": self.fp[c], "FN": self.fn[c]}
                      for c in self.class_names}
        return {
            "mIoU": self.mIoU(),
            "mDice": self.mDice(),
            "per_class": per_class,
        }

class PerPromptClassAccumulator:

    def __init__(self, class_names, prompts_per_class):
        self.class_names = list(class_names)
        self.prompts_per_class = {c: list(prompts_per_class[c]) for c in self.class_names}
        
        self.accs = {}
        for c, prompts in self.prompts_per_class.items():
            for p in prompts:
                self.accs[(c, p)] = ClassWiseAccumulator([c])

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray,
               class_name: str, prompt: str):
        self.accs[(class_name, prompt)].update(pred_mask, gt_mask, class_name)

    def summary(self) -> dict:
        per_class = {}
        for c in self.class_names:
            per_prompt = []
            for p in self.prompts_per_class[c]:
                acc = self.accs[(c, p)]
                per_prompt.append({
                    "prompt": p,
                    "IoU": acc.class_iou(c),
                    "Dice": acc.class_dice(c),
                    "TP": acc.tp[c], "FP": acc.fp[c], "FN": acc.fn[c],
                })
            ious = [pp["IoU"] for pp in per_prompt]
            dices = [pp["Dice"] for pp in per_prompt]
            per_class[c] = {
                "IoU": float(np.mean(ious)),   
                "Dice": float(np.mean(dices)),
                "per_prompt": per_prompt,
            }
        mIoU = float(np.mean([per_class[c]["IoU"] for c in self.class_names]))
        mDice = float(np.mean([per_class[c]["Dice"] for c in self.class_names]))
        return {"mIoU": mIoU, "mDice": mDice, "per_class": per_class}

def bootstrap_ci(values, n_boot: int = 1000, alpha: float = 0.05, seed: int = 0):
    if len(values) == 0:
        return 0.0, 0.0
    rng = np.random.default_rng(seed)
    vals = np.asarray(values, dtype=np.float64)
    boots = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
    lo = float(np.percentile(boots, 100 * alpha / 2))
    hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
    return lo, hi

In [ ]:
import sys
if "." not in sys.path: sys.path.insert(0, ".")
from pannuke_loader import PanNukeFold, DEFAULT_ROOT
from metrics import binary_dice
print("helpers loaded.")

## 01 — Build HoVer-Net (fast, nr_types=6) + load checkpoint

PanNuke = 5 cell types + background = `nr_types=6`. Weights come from the `'desc'`
key of the checkpoint (vqdang native format).

In [ ]:
from models.hovernet.net_desc import HoVerNet
from models.hovernet.post_proc import process

device = "cuda" if torch.cuda.is_available() else "cpu"
net = HoVerNet(input_ch=3, nr_types=6, mode="fast")
sd = torch.load(CKPT, map_location="cpu")
sd = sd["desc"] if isinstance(sd, dict) and "desc" in sd else sd
missing, unexpected = net.load_state_dict(sd, strict=False)
print(f"load_state_dict: {len(missing)} missing, {len(unexpected)} unexpected (want ~0/0)")
if missing[:3]:    print("  e.g. missing:", missing[:3])
if unexpected[:3]: print("  e.g. unexpected:", unexpected[:3])
net = net.to(device).eval()
print("HoVer-Net ready.")

## 02 — Inference fn (reflect-pad 256->384, center-crop output to 256)

Fast mode uses valid convolutions (output < input). We reflect-pad to 384 so the
valid output safely covers the original 256x256, then center-crop the output back to
256x256 (center-aligned → no need to know the exact border size). `NORM` is picked by
the smoke test below.

In [ ]:
NORM = "raw"  # HoVerNet.forward divides by 255 INTERNALLY -> feed raw 0-255 (smoke confirms)
_IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
_IMAGENET_STD  = np.array([0.229, 0.224, 0.225], np.float32)

def _prep(img256, norm):
    x = np.pad(img256, ((64, 64), (64, 64), (0, 0)), mode="reflect").astype(np.float32)  # 384
    if norm == "div255":
        x = x / 255.0
    elif norm == "imagenet":
        x = (x / 255.0 - _IMAGENET_MEAN) / _IMAGENET_STD
    # raw: leave as 0..255
    t = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0).to(device)
    return t

def _cc(a, size=256):
    h, w = a.shape[:2]
    s0, s1 = (h - size) // 2, (w - size) // 2
    return a[s0:s0 + size, s1:s1 + size]

@torch.no_grad()
def hover_predict(img256, norm=None, want_tp=False):
    norm = norm or NORM
    out = net(_prep(img256, norm))
    out = {k: v for k, v in out.items()}  # OrderedDict -> dict
    tp = torch.softmax(out["tp"], dim=1)[0].permute(1, 2, 0).cpu().numpy()   # H,W,6
    npr = torch.softmax(out["np"], dim=1)[0, 1].cpu().numpy()                # H,W  fg prob
    hv = out["hv"][0].permute(1, 2, 0).cpu().numpy()                         # H,W,2
    tp, npr, hv = _cc(tp), _cc(npr), _cc(hv)
    type_arg = tp.argmax(-1)[..., None].astype(np.float32)
    pred_map = np.concatenate([type_arg, npr[..., None], hv], axis=-1)        # H,W,4
    inst_map, inst_info = process(pred_map, nr_types=6)
    count = len(inst_info)
    fg = (npr > 0.5).astype(np.uint8)
    return (count, fg, tp) if want_tp else (count, fg)
print("inference fn ready.")

## 03 — Smoke: pick NORM (counts must be in a sane ballpark vs GT)

Try the 3 normalizations on a few Fold-1 images; the right one yields nucleus counts
in the same ballpark as GT (tens per patch), not ~0 or absurd.

In [ ]:
f1 = PanNukeFold(DEFAULT_ROOT, 1)
print(f"Fold 1: {len(f1)} images")
idxs = [0, 1, 2, 3, 4]
for norm in ["div255", "imagenet", "raw"]:
    rows = []
    for i in idxs:
        s = f1[i]
        c, _ = hover_predict(s["image"], norm=norm)
        rows.append((c, int(s["counts"].sum())))
    preds = [r[0] for r in rows]; gts = [r[1] for r in rows]
    mae = np.mean([abs(p - g) for p, g in zip(preds, gts)])
    print(f"  norm={norm:9s} preds={preds} gts={gts}  MAE={mae:.1f}")
print("\n-> set NORM below to the mode whose preds best track GT, then re-run cell 02.")

In [ ]:
# HoVerNet normalizes internally (imgs/255 inside forward) -> "raw" is correct.
# div255/imagenet will give ~0 nuclei (double-normalized black input).
NORM = "raw"
print("NORM =", NORM)

## 04 — Per-fold metrics (counting MAE + foreground Dice)

Subsample `PER_FOLD` images/fold for speed (the gap signal is clear well before all
images). The held-out fold shows **higher MAE / lower Dice**.

In [ ]:
from tqdm import tqdm
PER_FOLD = None   # None = ALL images/fold (tighter, ~40 min); or an int to subsample

def eval_fold(fold_id):
    fold = PanNukeFold(DEFAULT_ROOT, fold_id)
    if PER_FOLD is None:
        sel = np.arange(len(fold))
    else:
        sel = np.random.RandomState(0).permutation(len(fold))[:min(PER_FOLD, len(fold))]
    n = len(sel)
    abs_err, dices, gtc, prc = [], [], [], []
    for i in tqdm(sel, desc=f"Fold {fold_id}", leave=False):
        s = fold[int(i)]
        c, fg = hover_predict(s["image"])
        gt = int(s["counts"].sum())
        abs_err.append(abs(c - gt)); gtc.append(gt); prc.append(c)
        gt_fg = (s["masks"] > 0).any(0).astype(np.uint8)
        dices.append(binary_dice(fg, gt_fg))
    return {"n": n, "MAE": float(np.mean(abs_err)),
            "Dice": float(np.mean(dices)),
            "gt_mean": float(np.mean(gtc)), "pred_mean": float(np.mean(prc))}

results = {}
t0 = time.time()
for fid in [1, 2, 3]:
    results[fid] = eval_fold(fid)
    r = results[fid]
    print(f"Fold {fid}: MAE={r['MAE']:.2f} | Dice={r['Dice']:.3f} | "
          f"GT/pred mean={r['gt_mean']:.1f}/{r['pred_mean']:.1f} | n={r['n']}")
print(f"Done in {(time.time()-t0)/60:.1f} min")

## 05 — Verdict: which fold is held out?

In [ ]:
maes  = {f: results[f]["MAE"]  for f in results}
dices = {f: results[f]["Dice"] for f in results}
worst_mae  = max(maes,  key=maes.get)     # highest MAE  = most likely held-out
worst_dice = min(dices, key=dices.get)    # lowest Dice  = most likely held-out

print("=" * 60)
print("MEMORIZATION-GAP VERDICT")
print("=" * 60)
for f in [1, 2, 3]:
    print(f"  Fold {f}: MAE={maes[f]:.2f}  Dice={dices[f]:.3f}")
print("-" * 60)
mae_spread = (max(maes.values()) - min(maes.values()))
print(f"Worst (=> held-out candidate) by MAE : Fold {worst_mae}")
print(f"Worst (=> held-out candidate) by Dice: Fold {worst_dice}")
print(f"MAE spread across folds: {mae_spread:.2f}")
print()
if worst_mae == 3 and worst_dice == 3:
    print(">>> Fold 3 is clearly worst on BOTH metrics -> Fold 3 was HELD OUT -> SAFE to "
          "use this checkpoint for our Fold-3 conformal experiments.")
elif worst_mae == worst_dice:
    print(f">>> Fold {worst_mae} is the held-out one (both metrics agree), NOT Fold 3. "
          "Options: (a) switch our conformal set to this fold, or (b) retrain on the "
          "other two folds with Fold 3 held out.")
else:
    print(">>> Metrics DISAGREE or spread is small -> inconclusive / possibly trained on "
          "all folds -> treat as LEAKY -> retrain HoVer-Net on Fold 1+2 (Fold 3 held out).")

with open(f"{WORK}/hovernet_fold_verify.json", "w") as fjs:
    json.dump({"per_fold": results, "worst_by_mae": worst_mae,
               "worst_by_dice": worst_dice, "mae_spread": mae_spread, "norm": NORM}, fjs, indent=2)
print("\nSaved: hovernet_fold_verify.json")

## Notes

- A *clear* gap (Fold 3 worst by both MAE and Dice) is the green light. A small/ambiguous
  spread should be treated conservatively as leaky → retrain.
- If Fold 3 is confirmed held out, the next notebook reuses `hover_predict(..., want_tp=True)`
  to extract per-nucleus (s_i from np-branch over the instance, p_ik from the tp softmax)
  → build `hovernet_preds.pkl` in the same schema as `phase_C_preds_seed*.pkl` → re-run conformal.